# Lab 07 · A framework for choosing and checking

**Prepared by Mustafa Assaf** · 5–10 minute example

## Setup
Run this cell once. It loads the reusable helpers; the example follows below. Practice answers are scripted. For a live model, save `FREEINFERENCE_API_KEY` in Colab Secrets and change `MODE` to `"live"`.

In [ ]:
from importlib.util import find_spec
if not all(find_spec(name) for name in ['coursekit', 'langgraph']):
    %pip install -q "agentic-ai-coursekit[framework] @ https://github.com/mkassaf/agentic-ai-course/archive/aebb52597ab982f0b8b2f0832815b73dd6aca416.zip"
from coursekit import make_ask
MODE = "practice"
ask = make_ask(mode=MODE)

## Put the same two steps in LangGraph

A framework organizes steps and shared state. This real two-node LangGraph workflow asks for a choice, then checks it against our Python rule. Each node preserves the existing state before adding its result.

The model can still disagree with the policy. The framework makes that disagreement visible; it does not make the choice correct.

In [ ]:
from langgraph.graph import StateGraph, START, END
rule = 'Choose microservices only if team_size >= 8 AND independent_deploys is true; otherwise choose modular monolith.'
def choose(state):
    answer = ask(f"{rule} Case: {state['case']}. Return only the choice.", example="modular monolith").strip().lower()
    return {**state, "decision": answer}
def check(state):
    case = state["case"]
    expected = "microservices" if case["team_size"] >= 8 and case["independent_deploys"] else "modular monolith"
    return {**state, "matches_policy": state["decision"] == expected}
graph = StateGraph(dict)
graph.add_node("choose", choose)
graph.add_node("check", check)
graph.add_edge(START, "choose")
graph.add_edge("choose", "check")
graph.add_edge("check", END)
result = graph.compile().invoke({"case": {"team_size": 3, "independent_deploys": False}})
print(result["decision"], result["matches_policy"])
assert result["matches_policy"], "The model choice failed the policy check."

**Expected practice output:** `modular monolith True`. This is a fixed workflow, not an autonomous agent: the edges always choose, then check.

The check measures the same policy adherence as Lab 01. Real design review also needs evidence, alternatives, and trade-offs. Scripted practice output is not a model-quality benchmark.

**Small exercise:** try team size 8 with independent deployment `True`. Update the practice label to `microservices`, or run live. Confirm that the final check still passes.

## Read further

- [LangGraph overview](https://docs.langchain.com/oss/python/langgraph/overview)
- [LangGraph quickstart](https://docs.langchain.com/oss/python/langgraph/quickstart)